# Weather factors and check-in activity: distributions (T020 evidence)

## Goal

Show the **actual distribution** of the weather factors behind decision T020 (weather event
definitions and thresholds) and of check-in activity, for one metro and category. Every chart
uses the locally downloaded data: the staged Yelp files from
[notebook 01](01_yelp_dataset_exploration.ipynb) and the ERA5 weather from
[notebook 02](02_yelp_weather_dataset_exploration.ipynb).

Official thresholds are drawn on the charts so they can be judged against the local climate:

| Factor | Threshold | Source |
| --- | --- | --- |
| Daily rain | ≥ 10 mm (R10mm), ≥ 20 mm (R20mm), > wet-day p95 | [WMO/ETCCDI indices](https://etccdi.pacificclimate.org/list_27_indices.shtml) |
| Max temperature | ≥ 90 °F (32.2 °C) day; < 0 °C ice day | [NOAA 1991–2020 normals](https://www.ncei.noaa.gov/products/land-based-station/us-climate-normals), ETCCDI `ID` |
| Heat index | ≥ 96 °F (to 30 June) / ≥ 100 °F (from 1 July), 2 consecutive hours | [NWS Mount Holly criteria](https://www.weather.gov/phi/criteria), [NWS heat index equation](https://www.wpc.ncep.noaa.gov/html/heatindex_equation.shtml) |
| Wind chill | ≤ 20 °F (Code Blue); ≤ 0 °F (cold advisory, low end) | [City of Philadelphia Code Blue](https://www.phila.gov/codeBlue.html), [NWS winter criteria](https://www.weather.gov/phi/criteria-winter), [NWS wind chill](https://www.weather.gov/safety/cold-wind-chill-chart) |

Check-ins are **recorded activity**, not visitors or sales. Weather comparisons are
**associations**, not causal effects. Yelp check-in timestamps have no timezone; this notebook
treats them as UTC (`CHECKIN_TIMEZONE`, pending team confirmation) and section 5 shows the
evidence. Snowfall is not in the weather download, so snow is not analysed.

## Setup

From the repository root:

```sh
uv sync --frozen --group notebooks
uv run --frozen --group notebooks jupyter lab --notebook-dir=.
```

Run notebooks 01 and 02 first, then run these cells from top to bottom. Nothing is downloaded.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this repository or its notebooks directory.")
RAW_DIR = PROJECT_ROOT / "data/raw/yelp_exploration"
WEATHER_DIR = PROJECT_ROOT / "data/processed/yelp_weather"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/t020_weather_distribution"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CITY, STATE, CATEGORY = "Philadelphia", "PA", "Coffee & Tea"
LOCAL_TIMEZONE = "America/New_York"
CHECKIN_TIMEZONE = "UTC"  # assumption; see section 5
CLIMATE_START, CLIMATE_END = "2010-01-01", "2021-12-31"  # full years in the weather download
STUDY_START, STUDY_END = "2015-01-01", "2020-02-29"  # pre-COVID window for check-in comparisons
BASELINE_WEEKS = 4
MIN_EVENTS = 20
BOOTSTRAP = 2000
rng = np.random.default_rng(20261024)

GREEN, ORANGE, BLUE, GREY = "#26745a", "#eb6834", "#2a78d6", "#8a8f8c"
MONTHS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
    }
)


def f_to_c(value_f):
    return (value_f - 32) * 5 / 9


def c_to_f(value_c):
    return value_c * 9 / 5 + 32

## Load data

### Businesses and their weather cell

Businesses are filtered by city, state and an exact category match. Each business already has a
nearest ERA5 grid cell from notebook 02; the analysis uses the cell that contains most shops.

In [ ]:
business = pd.read_json(RAW_DIR / "business.jsonl", lines=True)
in_city = (business.city.str.strip().str.lower() == CITY.lower()) & (business.state == STATE)
has_category = business.categories.fillna("").map(
    lambda text: CATEGORY in [c.strip() for c in text.split(",")]
)
shops = business[in_city & has_category].reset_index(drop=True)
ids = set(shops.business_id)

mapping = pd.read_csv(WEATHER_DIR / "business_weather_mapping.csv")
mapping = mapping[mapping.business_id.isin(ids)]
cell_share = mapping.weather_cell_id.value_counts(normalize=True)
MAIN_CELL = cell_share.index[0]
cell_row = mapping[mapping.weather_cell_id == MAIN_CELL].iloc[0]

print(f"All businesses: {len(business):,}")
open_now = int(shops.is_open.sum())
print(f"{CATEGORY} in {CITY}, {STATE}: {len(shops):,} ({open_now} open in the 2022 snapshot)")
cell_point = (float(cell_row.weather_latitude), float(cell_row.weather_longitude))
print(f"Main weather cell: {MAIN_CELL} at {cell_point}, {cell_share.iloc[0]:.1%} of shops")
display(cell_share.rename("share of shops").to_frame())

### Check-ins → local days

Each check-in timestamp becomes one row. Timestamps are localised as `CHECKIN_TIMEZONE`, converted
to local time, and counted per local day for the whole metro (days with no check-ins are kept as
zeros).

In [ ]:
rows = []
for record in pd.read_json(RAW_DIR / "checkin.jsonl", lines=True, chunksize=50_000):
    chunk = record[record.business_id.isin(ids)]
    for business_id, dates in zip(chunk.business_id, chunk.date, strict=True):
        rows += [(business_id, ts.strip()) for ts in dates.split(",")]
checkins = pd.DataFrame(rows, columns=["business_id", "timestamp"])
checkins["timestamp"] = pd.to_datetime(checkins.timestamp, format="%Y-%m-%d %H:%M:%S")
checkins["local"] = (
    checkins.timestamp.dt.tz_localize(CHECKIN_TIMEZONE)
    .dt.tz_convert(LOCAL_TIMEZONE)
    .dt.tz_localize(None)
)
all_days = pd.date_range(checkins.local.min().normalize(), checkins.local.max().normalize())
daily_checkins = checkins.local.dt.normalize().value_counts().reindex(all_days, fill_value=0)
daily_checkins.index.name = "date"

in_study = checkins.local.between(STUDY_START, pd.Timestamp(STUDY_END) + pd.Timedelta(days=1))
first, last = checkins.local.min(), checkins.local.max()
print(f"Check-ins: {len(checkins):,} ({first:%Y-%m-%d} to {last:%Y-%m-%d})")
print(f"In the study window {STUDY_START} to {STUDY_END}: {int(in_study.sum()):,}")

### Hourly weather → heat index, wind chill and local days

* **Heat index** follows the NWS Weather Prediction Center equation (simple formula below 80 °F,
  Rothfusz regression with humidity adjustments above).
* **Wind chill** follows the NWS formula, valid at ≤ 50 °F with wind > 3 mph; otherwise it equals
  the air temperature. ERA5 wind is at 10 m, converted from m/s to mph.
* **Local days:** temperature for the local date of each hour; precipitation stamped at hour *t*
  fell during the preceding hour, so it counts toward the local date of *t − 1 h*. Days with fewer
  than 23 hours are dropped (23–25 hours occur around DST changes).

In [ ]:
frames = []
for path in sorted((WEATHER_DIR / "hourly").glob("era5_*.csv.gz")):
    for chunk in pd.read_csv(path, chunksize=500_000):
        frames.append(chunk[chunk.weather_cell_id == MAIN_CELL])
hourly = pd.concat(frames, ignore_index=True)
hourly["timestamp_utc"] = pd.to_datetime(hourly.timestamp_utc, utc=True)
hourly = hourly.sort_values("timestamp_utc").reset_index(drop=True)

t_f = c_to_f(hourly.temperature_2m)
rh = hourly.relative_humidity_2m
simple = 0.5 * (t_f + 61.0 + (t_f - 68.0) * 1.2 + rh * 0.094)
rothfusz = (
    -42.379
    + 2.04901523 * t_f
    + 10.14333127 * rh
    - 0.22475541 * t_f * rh
    - 6.83783e-3 * t_f**2
    - 5.481717e-2 * rh**2
    + 1.22874e-3 * t_f**2 * rh
    + 8.5282e-4 * t_f * rh**2
    - 1.99e-6 * t_f**2 * rh**2
)
low_rh = (rh < 13) & t_f.between(80, 112)
high_rh = (rh > 85) & t_f.between(80, 87)
rothfusz = rothfusz - np.where(
    low_rh, ((13 - rh) / 4) * np.sqrt(np.clip((17 - (t_f - 95).abs()) / 17, 0, None)), 0
)
rothfusz = rothfusz + np.where(high_rh, ((rh - 85) / 10) * ((87 - t_f) / 5), 0)
hourly["heat_index_f"] = np.where((simple + t_f) / 2 >= 80, rothfusz, simple)
wind_mph = hourly.wind_speed_10m * 2.23694
hourly["wind_chill_f"] = np.where(
    (t_f <= 50) & (wind_mph > 3),
    35.74 + 0.6215 * t_f - 35.75 * wind_mph**0.16 + 0.4275 * t_f * wind_mph**0.16,
    t_f,
)
hourly["local_date"] = (
    hourly.timestamp_utc.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None).dt.normalize()
)
hourly["rain_date"] = (
    (hourly.timestamp_utc - pd.Timedelta(hours=1))
    .dt.tz_convert(LOCAL_TIMEZONE)
    .dt.tz_localize(None)
    .dt.normalize()
)


def longest_run(flags):
    best = current = 0
    for flag in flags:
        current = current + 1 if flag else 0
        best = max(best, current)
    return best


grouped = hourly.groupby("local_date")
weather = grouped.agg(
    tmax=("temperature_2m", "max"),
    tmin=("temperature_2m", "min"),
    tmean=("temperature_2m", "mean"),
    rh_mean=("relative_humidity_2m", "mean"),
    wind_mean=("wind_speed_10m", "mean"),
    hi_max_f=("heat_index_f", "max"),
    wc_min_f=("wind_chill_f", "min"),
    hours=("temperature_2m", "count"),
)
weather["hi96_hours"] = grouped.heat_index_f.apply(lambda s: longest_run(s >= 96))
weather["hi100_hours"] = grouped.heat_index_f.apply(lambda s: longest_run(s >= 100))
rain = hourly.groupby("rain_date").agg(
    precip_mm=("precipitation", "sum"),
    rain_hours=("precipitation", "count"),
    max_hourly_mm=("precipitation", "max"),
)
weather = weather.join(rain, how="inner")
weather = weather[(weather.hours >= 23) & (weather.rain_hours >= 23)].drop(
    columns=["hours", "rain_hours"]
)
weather.index.name = "date"
climate = weather.loc[CLIMATE_START:CLIMATE_END]
print(
    f"Hourly rows for {MAIN_CELL}: {len(hourly):,}; complete local days: {len(weather):,}; "
    f"climate days {CLIMATE_START[:4]}–{CLIMATE_END[:4]}: {len(climate):,}"
)
display(climate.describe(percentiles=[0.05, 0.1, 0.5, 0.9, 0.95]).T.round(2))

## 1. Distribution of each factor

Histograms over the full years 2010–2021. Dashed lines are the official thresholds from the table
at the top. Precipitation uses **wet days only** (≥ 1 mm) on a log scale because most days are
dry. Heat index is shown for June–August, wind chill for December–February.

In [ ]:
wet = climate[climate.precip_mm >= 1].precip_mm
wet_p95 = float(wet.quantile(0.95))
summer = climate[climate.index.month.isin([6, 7, 8])]
winter = climate[climate.index.month.isin([12, 1, 2])]

panels = [
    (
        "Daily max temperature (°C)",
        climate.tmax,
        [(32.2, "90 °F day"), (0, "ice day < 0 °C")],
        False,
    ),
    ("Daily min temperature (°C)", climate.tmin, [(f_to_c(32), "freezing")], False),
    ("Daily mean temperature (°C)", climate.tmean, [], False),
    (
        "Daily rain on wet days (mm)",
        wet,
        [(10, "R10mm"), (20, "R20mm"), (wet_p95, f"p95 {wet_p95:.0f} mm")],
        True,
    ),
    ("Max heat index, Jun–Aug (°F)", summer.hi_max_f, [(96, "96 °F"), (100, "100 °F")], False),
    (
        "Min wind chill, Dec–Feb (°F)",
        winter.wc_min_f,
        [(20, "Code Blue 20 °F"), (0, "0 °F")],
        False,
    ),
    ("Mean relative humidity (%)", climate.rh_mean, [], False),
    ("Mean wind speed at 10 m (m/s)", climate.wind_mean, [], False),
    (
        "Wettest hour of the day (mm)",
        climate.max_hourly_mm[climate.max_hourly_mm >= 0.1],
        [(7.6, "AMS heavy rate")],
        True,
    ),
]
fig, axes = plt.subplots(3, 3, figsize=(13, 9.5), layout="constrained")
for ax, (label, values, lines, log_y) in zip(axes.flat, panels, strict=True):
    ax.hist(values.dropna(), bins=40, color=GREEN, alpha=0.85)
    top = ax.get_ylim()[1]
    for position, (x, text) in enumerate(lines):
        share = (values >= x).mean() if x > values.median() else (values <= x).mean()
        ax.axvline(x, color=ORANGE, linestyle="--", linewidth=1.3)
        height = top / (4**position) if log_y else top * (0.95 - 0.17 * position)
        ax.text(
            x,
            height,
            f" {text}\n {share:.1%} of days",
            color=ORANGE,
            fontsize=7.5,
            va="top",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1},
        )
    ax.set_title(label)
    ax.set_ylabel("days")
    if log_y:
        ax.set_yscale("log")
period = f"{CLIMATE_START[:4]}–{CLIMATE_END[:4]}"
fig.suptitle(f"Distribution of daily weather, {CITY} cell {MAIN_CELL}, {period}", fontsize=12)
plt.show()

percentiles = climate[
    ["tmax", "tmin", "tmean", "precip_mm", "hi_max_f", "wc_min_f", "rh_mean", "wind_mean"]
].quantile([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(percentiles.round(1))
print(
    f"Dry days (< 1 mm): {(climate.precip_mm < 1).mean():.1%}; wet-day p95 = {wet_p95:.1f} mm, "
    f"p99 = {wet.quantile(0.99):.1f} mm"
)

## 2. Seasonal pattern and local percentile thresholds

"Extreme" is relative to the local season. Box plots show each calendar month; the second chart
shows the ETCCDI-style local p10/p90 bands (± 2-day calendar window over 2010–2021) that the
percentile definitions use. A hot day in April and a hot day in July need different values to
count as unusual.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
by_month = climate.index.month
for ax, column, label in [
    (axes[0], "tmax", "Daily max temperature (°C)"),
    (axes[1], "tmin", "Daily min temperature (°C)"),
    (axes[2], "precip_mm", "Daily rain (mm)"),
]:
    data = [climate[column][by_month == m] for m in range(1, 13)]
    ax.boxplot(
        data,
        tick_labels=MONTHS,
        showfliers=column == "precip_mm",
        flierprops={"markersize": 2},
        medianprops={"color": ORANGE},
        boxprops={"color": GREEN},
        whiskerprops={"color": GREEN},
        capprops={"color": GREEN},
    )
    ax.set_title(label)
axes[0].axhline(32.2, color=ORANGE, linestyle="--", linewidth=1)
axes[1].axhline(0, color=BLUE, linestyle="--", linewidth=1)
axes[2].axhline(20, color=ORANGE, linestyle="--", linewidth=1)
axes[2].set_yscale("symlog", linthresh=1)
plt.show()


def calendar_day(index):
    day = index.dayofyear.to_numpy()
    shift = index.is_leap_year & (index.month > 2)
    feb29 = (index.month == 2) & (index.day == 29)
    return np.where(shift | feb29, day - 1, day)


days = calendar_day(climate.index)
bands = []
for day in range(1, 366):
    distance = np.abs(days - day)
    window = np.minimum(distance, 365 - distance) <= 2
    bands.append(
        {
            "day": day,
            "tmax_p10": np.percentile(climate.tmax[window], 10),
            "tmax_p50": np.percentile(climate.tmax[window], 50),
            "tmax_p90": np.percentile(climate.tmax[window], 90),
            "tmin_p10": np.percentile(climate.tmin[window], 10),
            "tmin_p90": np.percentile(climate.tmin[window], 90),
        }
    )
bands = pd.DataFrame(bands).set_index("day")
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(
    bands.index, bands.tmax_p10, bands.tmax_p90, color=ORANGE, alpha=0.18, label="max temp p10–p90"
)
ax.plot(
    bands.index, bands.tmax_p90, color=ORANGE, linewidth=1.5, label="max temp p90 (TX90p threshold)"
)
ax.fill_between(
    bands.index, bands.tmin_p10, bands.tmin_p90, color=BLUE, alpha=0.15, label="min temp p10–p90"
)
ax.plot(
    bands.index, bands.tmin_p10, color=BLUE, linewidth=1.5, label="min temp p10 (TN10p threshold)"
)
ax.axhline(32.2, color=GREY, linestyle="--", linewidth=1, label="90 °F (32.2 °C)")
ax.axhline(0, color=GREY, linestyle=":", linewidth=1, label="0 °C")
ax.set_xticks([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335], MONTHS)
ax.set_ylabel("°C")
ax.set_title("Local percentile thresholds by calendar day (± 2-day window, 2010–2021)")
ax.legend(fontsize=8, ncols=3, loc="lower center")
plt.show()

## 3. How often each candidate event happens

Each definition becomes a yes/no flag per local day. The heatmap counts event days per year; the
table gives the mean per year and the number of days in the pre-COVID study window, which is the
sample available for check-in comparisons (the team rule needs at least 20).

In [ ]:
cal = pd.Series(calendar_day(weather.index), index=weather.index)
tmax_p90 = cal.map(bands.tmax_p90)
tmin_p10 = cal.map(bands.tmin_p10)
early_summer = weather.index.month < 7
DEFINITIONS = {
    "Rain ≥ 10 mm (R10mm)": weather.precip_mm >= 10,
    "Rain ≥ 20 mm (R20mm)": weather.precip_mm >= 20,
    f"Rain > wet-day p95 ({wet_p95:.0f} mm)": weather.precip_mm > wet_p95,
    "Any hour ≥ 7.6 mm (AMS rate)": weather.max_hourly_mm >= 7.6,
    "Max temp ≥ 90 °F (32.2 °C)": weather.tmax >= 32.2,
    "Max temp > local p90 (TX90p)": weather.tmax > tmax_p90,
    "NWS heat advisory (HI 96/100 °F, 2 h)": (early_summer & (weather.hi96_hours >= 2))
    | (~early_summer & (weather.hi100_hours >= 2)),
    "Code Blue (wind chill ≤ 20 °F)": weather.wc_min_f <= 20,
    "Wind chill ≤ 0 °F": weather.wc_min_f <= 0,
    "Ice day (max temp < 0 °C)": weather.tmax < 0,
    "Min temp < local p10 (TN10p)": weather.tmin < tmin_p10,
}
flags = pd.DataFrame(DEFINITIONS)
per_year = (
    flags.loc[CLIMATE_START:CLIMATE_END]
    .groupby(flags.loc[CLIMATE_START:CLIMATE_END].index.year)
    .sum()
)

fig, ax = plt.subplots(figsize=(12, 4.6))
image = ax.imshow(per_year.T.to_numpy(), aspect="auto", cmap="Greens")
ax.set_yticks(range(len(per_year.columns)), per_year.columns, fontsize=8)
ax.set_xticks(range(len(per_year.index)), per_year.index, fontsize=8)
ax.grid(False)
for (row, col), value in np.ndenumerate(per_year.T.to_numpy()):
    ax.text(
        col,
        row,
        int(value),
        ha="center",
        va="center",
        fontsize=7,
        color="white" if value > per_year.to_numpy().max() * 0.6 else "black",
    )
ax.set_title("Event days per year")
fig.colorbar(image, ax=ax, fraction=0.02)
plt.show()

frequency = pd.DataFrame(
    {
        "days per year (mean)": per_year.mean().round(1),
        "min year": per_year.min(),
        "max year": per_year.max(),
        "days in study window": flags.loc[STUDY_START:STUDY_END].sum(),
    }
)
frequency["meets ≥ 20 rule"] = frequency["days in study window"] >= MIN_EVENTS
display(frequency)

## 4. Is the ERA5 grid cell representative? Comparison with NOAA

NOAA's 1991–2020 normals for Philadelphia International Airport (station `USW00013739`) give
official average day counts. Values below were read from NOAA's
[annual normals service](https://www.ncei.noaa.gov/access/services/data/v1?dataset=normals-annualseasonal-1991-2020&stations=USW00013739&format=json)
on 2026-10-07 (fields `ANN-TMAX-AVGNDS-GRTH090`, `ANN-TMAX-AVGNDS-LSTH032`,
`ANN-PRCP-AVGNDS-GE050HI`, `ANN-PRCP-AVGNDS-GE100HI`). The periods differ (1991–2020 station vs
2010–2021 grid cell), and a 0.25° cell smooths local extremes, so close but not identical values
are expected.

In [ ]:
noaa = {
    "Max temp ≥ 90 °F": (29.7, climate.tmax >= 32.2),
    "Max temp < 32 °F (ice day)": (13.1, climate.tmax < 0),
    "Rain ≥ 0.50 in (12.7 mm)": (29.5, climate.precip_mm >= 12.7),
    "Rain ≥ 1.00 in (25.4 mm)": (10.9, climate.precip_mm >= 25.4),
}
comparison = pd.DataFrame(
    {
        "NOAA PHL 1991–2020": {k: v[0] for k, v in noaa.items()},
        f"ERA5 {MAIN_CELL} 2010–2021": {
            k: v[1].groupby(v[1].index.year).sum().mean() for k, v in noaa.items()
        },
    }
).round(1)
display(comparison)
ax = comparison.plot.barh(color=[GREY, GREEN], figsize=(8, 3.2), width=0.75)
ax.set_xlabel("days per year")
ax.set_title("Official station normals vs the grid cell used here")
ax.invert_yaxis()
plt.show()

## 5. Distribution of check-in activity

Yelp check-in use rises and then falls sharply over the years (and collapses in 2020), so raw
counts are not comparable across years; later comparisons use nearby weeks only. The hour-of-day
chart compares the raw (naive) timestamps with the UTC → local conversion: a coffee-shop profile
peaking around midday supports the UTC assumption.

In [ ]:
study_days = daily_checkins.loc[STUDY_START:STUDY_END]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
yearly = daily_checkins.groupby(daily_checkins.index.year).mean()
axes[0, 0].bar(
    yearly.index.astype(str), yearly, color=[ORANGE if y == 2020 else GREEN for y in yearly.index]
)
axes[0, 0].set_title("Average check-ins per day, by year")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].hist(study_days, bins=40, color=GREEN)
axes[0, 1].set_title(f"Daily check-ins, {STUDY_START[:4]}–{STUDY_END[:7]}")
axes[0, 1].set_xlabel("check-ins per day (metro)")

weekday_mean = study_days.groupby(study_days.index.weekday).mean()
axes[0, 2].bar(WEEKDAYS, weekday_mean, color=GREEN)
axes[0, 2].set_title("Average by weekday (study window)")

month_mean = study_days.groupby(study_days.index.month).mean()
axes[1, 0].bar(month_mean.index, month_mean, color=GREEN)
axes[1, 0].set_xticks(range(1, 13), MONTHS)
axes[1, 0].set_title("Average by month (study window)")

naive_hours = checkins.timestamp.dt.hour.value_counts(normalize=True).sort_index() * 100
local_hours = checkins.local.dt.hour.value_counts(normalize=True).sort_index() * 100
axes[1, 1].plot(
    naive_hours.index, naive_hours, color=GREY, marker="o", markersize=3, label="raw timestamps"
)
axes[1, 1].plot(
    local_hours.index, local_hours, color=GREEN, marker="o", markersize=3, label="as UTC → local"
)
axes[1, 1].set_xticks(range(0, 24, 3))
axes[1, 1].set_title("Share of check-ins by hour of day (%)")
axes[1, 1].legend(fontsize=8)

study_checkins = checkins[in_study]
matrix = (
    study_checkins.groupby([study_checkins.local.dt.weekday, study_checkins.local.dt.hour])
    .size()
    .unstack(fill_value=0)
)
axes[1, 2].imshow(
    matrix.reindex(index=range(7), columns=range(24), fill_value=0), aspect="auto", cmap="Greens"
)
axes[1, 2].set_yticks(range(7), WEEKDAYS)
axes[1, 2].set_xticks(range(0, 24, 3))
axes[1, 2].grid(False)
axes[1, 2].set_title("Check-ins by weekday and local hour")
plt.show()
print(f"Peak hour raw: {naive_hours.idxmax()}h; after UTC → local: {local_hours.idxmax()}h")

## 6. Weather against check-in activity

For every day in the study window, **relative activity** = that day's check-ins ÷ the mean of the
same weekday in the ± 4 surrounding weeks (excluding days flagged by any definition above),
minus 1. This removes the long-term Yelp trend, the weekday cycle and most of the season. The
binned charts show the mean relative activity per weather bin with a 95% bootstrap interval;
grey numbers are the days in each bin.

In [ ]:
any_event = flags.any(axis=1)
counts = daily_checkins
dates = study_days.index
relative = {}
for day in dates:
    base = [day + pd.Timedelta(weeks=k) for k in range(-BASELINE_WEEKS, BASELINE_WEEKS + 1) if k]
    base = [b for b in base if b in study_days.index and not any_event.get(b, False)]
    if len(base) >= 3 and counts[base].mean() > 0:
        relative[day] = (counts[day], counts[base].mean())
relative = pd.DataFrame(relative, index=["observed", "expected"]).T
relative["change_pct"] = (relative.observed / relative.expected - 1) * 100
relative = relative.join(weather, how="inner")
print(f"Days with a usable baseline: {len(relative):,}")


def binned(ax, column, edges, label, threshold=None):
    groups = pd.cut(relative[column], edges)
    stats = []
    for interval, group in relative.groupby(groups, observed=True):
        if len(group) < 5:
            continue
        obs, exp = group.observed.to_numpy(), group.expected.to_numpy()
        picks = rng.integers(0, len(obs), (BOOTSTRAP, len(obs)))
        boot = (obs[picks].sum(1) / exp[picks].sum(1) - 1) * 100
        name = f"{interval.left:g}–{interval.right:g}"
        stats.append(
            (
                interval.left,
                (obs.sum() / exp.sum() - 1) * 100,
                *np.percentile(boot, [2.5, 97.5]),
                len(group),
                name,
            )
        )
    stats = pd.DataFrame(stats, columns=["left", "effect", "low", "high", "n", "label"])
    x = np.arange(len(stats))
    ax.axhline(0, color=GREY, linewidth=1)
    ax.errorbar(
        x,
        stats.effect,
        yerr=[stats.effect - stats.low, stats.high - stats.effect],
        fmt="o-",
        color=GREEN,
        ecolor=GREEN,
        capsize=3,
        markersize=4,
    )
    for xi, row in zip(x, stats.itertuples(), strict=True):
        ax.text(xi, row.high + 1, int(row.n), ha="center", fontsize=7, color=GREY)
    if threshold is not None:
        # dashed line between the bins below and above the threshold
        ax.axvline(
            int(np.searchsorted(stats.left.to_numpy(), threshold)) - 0.5,
            color=ORANGE,
            linestyle="--",
            linewidth=1.2,
            label=f"threshold {threshold:g}",
        )
        ax.legend(fontsize=7, loc="lower left")
    ax.set_xticks(x, stats.label, rotation=35, ha="right", fontsize=7.5)
    ax.set_xlabel(label)
    ax.set_ylabel("check-in change vs baseline (%)")


fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), layout="constrained")
binned(axes[0, 0], "precip_mm", [-0.1, 0.5, 2, 5, 10, 15, 20, 30, 200], "daily rain (mm)", 20)
binned(axes[0, 1], "tmax", np.arange(-10, 40, 4), "daily max temperature (°C)", 32.2)
binned(axes[0, 2], "tmin", np.arange(-20, 30, 4), "daily min temperature (°C)", 0)
binned(axes[1, 0], "hi_max_f", [80, 85, 90, 95, 100, 105, 120], "max heat index (°F)", 100)
binned(axes[1, 1], "wc_min_f", [-20, 0, 10, 20, 30, 40, 50], "min wind chill (°F)", 20)
binned(axes[1, 2], "wind_mean", np.arange(0, 12, 1.5), "mean wind speed (m/s)")
fig.suptitle("Check-in change by weather bin (association, not causation)", fontsize=12)
plt.show()

### Effect of each event definition, under two baseline rules

Event days are compared with the same weekday in the ± 4 surrounding weeks. Which days may serve
as a baseline is a **method choice that changes the numbers**, so both are shown:

* **same family:** baseline days exclude days flagged by any definition of the same kind (rain,
  heat or cold);
* **strict:** baseline days exclude days flagged by any definition. Frequent definitions such as
  Code Blue (about 40 winter days a year) then remove many cold-season baselines, so fewer event
  days remain usable.

A definition is robust when both rules agree. Bars show the *same family* rule with a 95%
bootstrap interval; the black dot is the *strict* rule. Grey = fewer than 20 usable event days.

In [ ]:
FAMILIES = {
    "rain": [k for k in DEFINITIONS if k.startswith(("Rain", "Any hour"))],
    "heat": [k for k in DEFINITIONS if k.startswith(("Max temp", "NWS heat"))],
    "cold": [
        k for k in DEFINITIONS if k.startswith(("Code Blue", "Wind chill", "Ice day", "Min temp"))
    ],
}
family_of = {name: family for family, names in FAMILIES.items() for name in names}
study_index = study_days.index


def effect(event_mask, exclude_mask):
    event_mask = event_mask.reindex(study_index, fill_value=False)
    exclude = exclude_mask.reindex(study_index, fill_value=False)
    obs, exp = [], []
    for day in study_index[event_mask.to_numpy()]:
        base = [
            day + pd.Timedelta(weeks=k) for k in range(-BASELINE_WEEKS, BASELINE_WEEKS + 1) if k
        ]
        base = [b for b in base if b in study_index and not exclude[b]]
        if len(base) >= 3 and counts[base].mean() > 0:
            obs.append(counts[day])
            exp.append(counts[base].mean())
    obs, exp = np.array(obs, float), np.array(exp, float)
    if len(obs) < 3:
        return len(obs), np.nan, np.nan, np.nan
    picks = rng.integers(0, len(obs), (BOOTSTRAP, len(obs)))
    boot = (obs[picks].sum(1) / exp[picks].sum(1) - 1) * 100
    low, high = np.percentile(boot, [2.5, 97.5])
    return len(obs), (obs.sum() / exp.sum() - 1) * 100, low, high


rows = []
for name in DEFINITIONS:
    family_mask = flags[FAMILIES[family_of[name]]].any(axis=1)
    n_f, eff_f, low_f, high_f = effect(flags[name], family_mask)
    n_s, eff_s, low_s, high_s = effect(flags[name], any_event)
    rows.append(
        {
            "definition": name,
            "family": family_of[name],
            "n (same family)": n_f,
            "change % (same family)": eff_f,
            "ci_low": low_f,
            "ci_high": high_f,
            "n (strict)": n_s,
            "change % (strict)": eff_s,
            "ci_low strict": low_s,
            "ci_high strict": high_s,
        }
    )
results = pd.DataFrame(rows)
results["clear (same family)"] = (results["n (same family)"] >= MIN_EVENTS) & (
    (results.ci_low > 0) | (results.ci_high < 0)
)
results["clear (strict)"] = (results["n (strict)"] >= MIN_EVENTS) & (
    (results["ci_low strict"] > 0) | (results["ci_high strict"] < 0)
)

plot = results.dropna(subset=["change % (same family)"]).iloc[::-1].reset_index(drop=True)
labels = [f"{d}  (n={n})" for d, n in zip(plot.definition, plot["n (same family)"], strict=True)]
colors = [
    GREY if n < MIN_EVENTS else (ORANGE if c else GREEN)
    for n, c in zip(plot["n (same family)"], plot["clear (same family)"], strict=True)
]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(plot) + 1.2))
y = np.arange(len(plot))
ax.barh(y, plot["change % (same family)"], color=colors, height=0.55)
ax.errorbar(
    plot["change % (same family)"],
    y,
    xerr=[
        plot["change % (same family)"] - plot.ci_low,
        plot.ci_high - plot["change % (same family)"],
    ],
    fmt="none",
    ecolor="black",
    elinewidth=1,
    capsize=3,
)
ax.scatter(plot["change % (strict)"], y, color="black", s=22, zorder=3, label="strict rule")
ax.set_yticks(y, labels, fontsize=8.5)
ax.axvline(0, color=GREY, linewidth=1)
ax.set_xlabel("check-in change on event days vs baseline (%)")
ax.set_title(
    "Orange = clear change (n ≥ 20, CI excludes 0) · green = no clear change · grey = n < 20"
)
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8, loc="lower left")
plt.show()
display(results.round(1))

## Save summaries

Small CSV summaries for the report and for review. Raw data stays where notebooks 01 and 02 put
it; all of these files are ignored by Git.

In [ ]:
percentiles.round(3).to_csv(OUTPUT_DIR / "weather_percentiles.csv")
frequency.to_csv(OUTPUT_DIR / "event_frequency.csv")
comparison.to_csv(OUTPUT_DIR / "noaa_comparison.csv")
results.round(3).to_csv(OUTPUT_DIR / "event_effects.csv", index=False)
display(pd.DataFrame({"output": sorted(p.name for p in OUTPUT_DIR.iterdir())}))

## Reading the results

* Use sections 1–2 to see where each threshold sits in Philadelphia's own climate, not a generic
  number. Section 4 checks that the grid cell represents the city's official station.
* An event definition is usable for T020 when it comes from an authority for this place, has at
  least 20 days in the study window (section 3), and shows a clear change (section 6).
* These are metro-wide associations for one category, under the UTC timestamp assumption, and
  thresholds were examined on the same period they are compared on. Confirm the timezone and use a
  separate period before training any model on these definitions.